# Feature engineering enriched

## Read csv

In [61]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
import joblib
import os


In [62]:
df_clean = pd.read_csv("../data/OpenAlex/openalex_ai_papers_enriched_cleaned.csv")
df_clean.head()

,id,title,publication_year,language,cited_by_count,referenced_works_count,fwci,authorship_count,countries_distinct_count,publication_type,...,sdg_8,sdg_9,sdg_10,sdg_11,sdg_12,sdg_13,sdg_14,sdg_15,sdg_16,sdg_17
0,https://openalex.org/W1902237438,Effective Approaches to Attention-based Neural...,2015,en,8508,20,633.3691,3,1,proceedings-article,...,0,0,0,0,0,0,0,0,0,0
1,https://openalex.org/W658020064,From Word Embeddings To Document Distances,2015,en,1518,47,164.8113,4,1,other,...,0,0,0,0,0,0,0,0,0,0
2,https://openalex.org/W2251135946,Distant Supervision for Relation Extraction vi...,2015,en,1200,31,87.4904,4,1,proceedings-article,...,0,0,0,0,0,0,0,0,0,0
3,https://openalex.org/W1491002046,An Introduction to Recursive Partitioning Usin...,2015,en,1030,7,105.2714,2,0,unknown,...,0,0,0,0,0,0,0,0,0,0
4,https://openalex.org/W2250342921,chrF: character n-gram F-score for automatic M...,2015,en,935,10,36.6006,1,1,proceedings-article,...,0,0,0,0,0,0,0,0,0,0


In [45]:
print(df_clean.shape)
print(df_clean.dtypes)

(293045, 44)
id                            object
title                         object
publication_year               int64
language                      object
cited_by_count                 int64
referenced_works_count         int64
fwci                         float64
authorship_count               int64
countries_distinct_count       int64
publication_type              object
is_oa                          int64
oa_status                     object
citation_top_1_percent         int64
citation_top_10_percent        int64
keyword_count                  int64
primary_topic_score          float64
first_year_citations           int64
topic_id                      object
topic_name                    object
unique_authors_count         float64
unique_institutions_count    float64
institution_edu_count        float64
funder_count                 float64
award_count                  float64
sdg_count                    float64
sdg_max_score                float64
sdg_display_names        

In [57]:
import pandas as pd

# Check only numerical features that could have meaningful zeros
check_cols = [
    'referenced_works_count',
    'authorship_count',
    'countries_distinct_count',
    'keyword_count',
    'primary_topic_score',
    'unique_authors_count',
    'unique_institutions_count',
    'institution_edu_count',
    'funder_count',
    'award_count',
    'sdg_count',
    'sdg_4'
]

zero_summary = pd.DataFrame({
    'zero_count': (df_clean[check_cols] == 0).sum(),
    'zero_pct': ((df_clean[check_cols] == 0).mean() * 100).round(1)
}).sort_values('zero_pct', ascending=False)

print(zero_summary.to_string())

                           zero_count  zero_pct
award_count                    237376      81.0
sdg_4                          220786      75.3
funder_count                   216456      73.9
sdg_count                      143287      48.9
institution_edu_count           89860      30.7
countries_distinct_count        74428      25.4
unique_institutions_count       73746      25.2
referenced_works_count          72778      24.8
unique_authors_count             1737       0.6
authorship_count                 1702       0.6
keyword_count                     272       0.1
primary_topic_score                 0       0.0


## Drop leaky features

Now we will drop the columns that are not used for modeling - leaky or non needed.

In [46]:
cols_to_drop = [
    'id',
    'title',
    'cited_by_count',
    'fwci',
    'citation_top_1_percent',
    'first_year_citations',
    'sdg_display_names',  # EDA only column
    'topic_id', 
]

df_model = df_clean.drop(columns=cols_to_drop)
print(df_model.shape)
df_model.columns.tolist()

(293045, 36)


['publication_year',
 'language',
 'referenced_works_count',
 'authorship_count',
 'countries_distinct_count',
 'publication_type',
 'is_oa',
 'oa_status',
 'citation_top_10_percent',
 'keyword_count',
 'primary_topic_score',
 'topic_name',
 'unique_authors_count',
 'unique_institutions_count',
 'institution_edu_count',
 'funder_count',
 'award_count',
 'sdg_count',
 'sdg_max_score',
 'sdg_1',
 'sdg_2',
 'sdg_3',
 'sdg_4',
 'sdg_5',
 'sdg_6',
 'sdg_7',
 'sdg_8',
 'sdg_9',
 'sdg_10',
 'sdg_11',
 'sdg_12',
 'sdg_13',
 'sdg_14',
 'sdg_15',
 'sdg_16',
 'sdg_17']

## Language column

Let's put all non-English languages to category `other`

In [47]:
df_model['language'] = df_model['language'].where(
    df_model['language'] == 'en', other='other'
)

print(df_model['language'].value_counts())

language
en       279468
other     13577
Name: count, dtype: int64


Separate target

In [48]:
target = 'citation_top_10_percent'

X = df_model.drop(columns=[target])
y = df_model[target]

print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')
print(f'Target distribution:\n{y.value_counts(normalize=True).round(3)}')

X shape: (293045, 35)
y shape: (293045,)
Target distribution:
citation_top_10_percent
0    0.82
1    0.18
Name: proportion, dtype: float64


Check for nulls

In [49]:
null_counts = X.isnull().sum()
null_counts = null_counts[null_counts > 0]

if len(null_counts) == 0:
    print('No nulls found — ready to split')
else:
    print('Nulls found:')
    print(null_counts)

No nulls found — ready to split


## Numerical/Categorical features

In [50]:
categorical_cols = [
    'publication_type',
    'oa_status',
    'topic_name',
    'language'
]

numerical_cols = [
    'publication_year',
    'authorship_count',
    'countries_distinct_count',
    'referenced_works_count',
    'keyword_count',
    'primary_topic_score',
    'is_oa',
    # New enriched features
    'unique_authors_count',
    'unique_institutions_count',
    'institution_edu_count',
    'funder_count',
    'award_count',
    'sdg_count',
    'sdg_max_score',
    'sdg_1', 'sdg_2', 'sdg_3', 'sdg_4', 'sdg_5',
    'sdg_6', 'sdg_7', 'sdg_8', 'sdg_9', 'sdg_10',
    'sdg_11', 'sdg_12', 'sdg_13', 'sdg_14', 'sdg_15',
    'sdg_16', 'sdg_17'
]

print('Categorical columns:', len(categorical_cols))
print('Numerical columns:', len(numerical_cols))
print('Total features:', len(categorical_cols) + len(numerical_cols))

Categorical columns: 4
Numerical columns: 31
Total features: 35


## Train/Test split

In [51]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y) 
# stratify=y ensures the 82/18 class balance is preserved in both train and test sets.

In [52]:
print(f'X_train_enriched shape: {X_train.shape}')
print(f'X_test_enriched shape:  {X_test.shape}')
print(f'y_train_enriched distribution:\n{y_train.value_counts(normalize=True).round(3)}')
print(f'y_test_enriched distribution:\n{y_test.value_counts(normalize=True).round(3)}')

X_train_enriched shape: (234436, 35)
X_test_enriched shape:  (58609, 35)
y_train_enriched distribution:
citation_top_10_percent
0    0.82
1    0.18
Name: proportion, dtype: float64
y_test_enriched distribution:
citation_top_10_percent
0    0.82
1    0.18
Name: proportion, dtype: float64


## One-hot encode categoricals

In [53]:
# Initialise encoder — handle_unknown='ignore' encodes unseen categories as all zeros
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

# Fit on train only to avoid data leakage
ohe.fit(X_train[categorical_cols])

# Transform both train and test using the fitted encoder
encoded_train = ohe.transform(X_train[categorical_cols])
encoded_test = ohe.transform(X_test[categorical_cols])

# Get the new column names from the encoder
encoded_col_names = ohe.get_feature_names_out(categorical_cols)

# Combine numerical columns with encoded categorical columns
X_train_encoded = pd.concat([
    X_train[numerical_cols].reset_index(drop=True),
    pd.DataFrame(encoded_train, columns=encoded_col_names)
], axis=1)

X_test_encoded = pd.concat([
    X_test[numerical_cols].reset_index(drop=True),
    pd.DataFrame(encoded_test, columns=encoded_col_names)
], axis=1)

print(f'X_train_enriched_encoded shape: {X_train_encoded.shape}')
print(f'X_test_enriched_encoded shape:  {X_test_encoded.shape}')
X_train_encoded.columns.tolist()

X_train_enriched_encoded shape: (234436, 54)
X_test_enriched_encoded shape:  (58609, 54)


['publication_year',
 'authorship_count',
 'countries_distinct_count',
 'referenced_works_count',
 'keyword_count',
 'primary_topic_score',
 'is_oa',
 'unique_authors_count',
 'unique_institutions_count',
 'institution_edu_count',
 'funder_count',
 'award_count',
 'sdg_count',
 'sdg_max_score',
 'sdg_1',
 'sdg_2',
 'sdg_3',
 'sdg_4',
 'sdg_5',
 'sdg_6',
 'sdg_7',
 'sdg_8',
 'sdg_9',
 'sdg_10',
 'sdg_11',
 'sdg_12',
 'sdg_13',
 'sdg_14',
 'sdg_15',
 'sdg_16',
 'sdg_17',
 'publication_type_journal-article',
 'publication_type_other',
 'publication_type_proceedings-article',
 'publication_type_thesis',
 'publication_type_unknown',
 'oa_status_bronze',
 'oa_status_closed',
 'oa_status_diamond',
 'oa_status_gold',
 'oa_status_green',
 'oa_status_hybrid',
 'topic_name_Anomaly Detection Techniques and Applications',
 'topic_name_Evolutionary Algorithms and Applications',
 'topic_name_Metaheuristic Optimization Algorithms Research',
 'topic_name_Natural Language Processing Techniques',
 'topic

## Saving features

In [54]:
# Create output directories if they don't exist
os.makedirs('../data/features', exist_ok=True)
os.makedirs('../models', exist_ok=True)

# Save encoded feature matrices
X_train_encoded.to_csv('../data/features/X_train_enriched.csv', index=False)
X_test_encoded.to_csv('../data/features/X_test_enriched.csv', index=False)

# Save target vectors — reset index to match encoded feature matrices
y_train.reset_index(drop=True).to_csv('../data/features/y_train_enriched.csv', index=False)
y_test.reset_index(drop=True).to_csv('../data/features/y_test_enriched.csv', index=False)

# Save fitted encoder for use on new data in modelling and dashboard
joblib.dump(ohe, '../models/ohe_enriched.pkl')

print('Saved:')
print('  ../data/features/X_train_enriched.csv')
print('  ../data/features/X_test_enriched.csv')
print('  ../data/features/y_train_enriched.csv')
print('  ../data/features/y_test_enriched.csv')
print('  ../models/ohe_enriched.pkl')


Saved:
  ../data/features/X_train_enriched.csv
  ../data/features/X_test_enriched.csv
  ../data/features/y_train_enriched.csv
  ../data/features/y_test_enriched.csv
  ../models/ohe_enriched.pkl
